# 02 - Preprocessing

Ноутбук для базовой очистки данных, создания признаков и split train/val/test.

In [1]:
from pathlib import Path
from typing import Optional, Tuple

import pandas as pd
from sklearn.model_selection import train_test_split

DEFAULT_TARGET = "AdoptionSpeed"
DEFAULT_RANDOM_STATE = 42

In [2]:
def load_raw_data(csv_path: str) -> pd.DataFrame:
    return pd.read_csv(csv_path)


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    clean_df = df.copy().drop_duplicates()
    if "Name" in clean_df.columns:
        clean_df["NameLength"] = clean_df["Name"].fillna("").astype(str).str.len()
    return clean_df


def split_features_target(
    df: pd.DataFrame, target_col: str = DEFAULT_TARGET
) -> Tuple[pd.DataFrame, pd.Series]:
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' was not found.")
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y


def _build_stratify_target(y: pd.Series, use_stratify: bool) -> Optional[pd.Series]:
    if not use_stratify:
        return None
    value_counts = y.value_counts()
    if (value_counts < 2).any():
        return None
    return y


def make_train_val_test(
    df: pd.DataFrame,
    target_col: str = DEFAULT_TARGET,
    test_size: float = 0.2,
    val_size: float = 0.2,
    random_state: int = DEFAULT_RANDOM_STATE,
    use_stratify: bool = True,
):
    X, y = split_features_target(df=df, target_col=target_col)
    stratify_all = _build_stratify_target(y=y, use_stratify=use_stratify)

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_all,
    )

    val_ratio_in_train_full = val_size / (1 - test_size)
    stratify_train = _build_stratify_target(y=y_train_full, use_stratify=use_stratify)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=val_ratio_in_train_full,
        random_state=random_state,
        stratify=stratify_train,
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

In [3]:
TRAIN_PATH = Path("../data/raw/train/train.csv")

if TRAIN_PATH.exists():
    raw_df = load_raw_data(str(TRAIN_PATH))
    clean_df = clean_dataframe(raw_df)
    X_train, X_val, X_test, y_train, y_val, y_test = make_train_val_test(clean_df)

    print("Raw shape:", raw_df.shape)
    print("Clean shape:", clean_df.shape)
    print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)
else:
    print(f"Файл не найден: {TRAIN_PATH.resolve()}")

Raw shape: (14993, 24)
Clean shape: (14993, 25)
Train/Val/Test: (8995, 24) (2999, 24) (2999, 24)


## 1) Поиск и источник данных

- Источник: соревнование Kaggle **PetFinder.my Adoption Prediction**.
- Почему выбран: реальная прикладная задача классификации с табличными и текстовыми признаками.
- Цель: предсказать `AdoptionSpeed` (класс скорости пристройства животного).

In [ ]:
import matplotlib.pyplot as plt

# Базовое описание датасета
print("Rows, cols:", raw_df.shape)
print("\nTarget distribution (AdoptionSpeed):")
print(raw_df[DEFAULT_TARGET].value_counts().sort_index())

print("\nDtypes:")
print(raw_df.dtypes.value_counts())

## 2) Полная очистка: пропуски, дубли, выбросы, типы

In [ ]:
# Пропуски
missing_abs = raw_df.isna().sum().sort_values(ascending=False)
missing_pct = (raw_df.isna().mean() * 100).sort_values(ascending=False)

missing_table = (
    pd.DataFrame({"missing_count": missing_abs, "missing_pct": missing_pct})
    .query("missing_count > 0")
)
print("Columns with missing values:")
missing_table.head(20)

In [ ]:
# Дубликаты
n_duplicates = raw_df.duplicated().sum()
print(f"Exact duplicate rows: {n_duplicates}")

# Типы и приведение категориальных полей к string для стабильного пайплайна
clean_df = raw_df.copy().drop_duplicates()

cat_cols = clean_df.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    clean_df[col] = clean_df[col].astype("string")

print("\nDtypes after normalization:")
print(clean_df.dtypes.value_counts())

In [ ]:
# Выбросы (IQR-оценка для числовых признаков)
num_cols = clean_df.select_dtypes(include=["number"]).columns.tolist()
num_cols_wo_target = [c for c in num_cols if c != DEFAULT_TARGET]

outlier_summary = []
for col in num_cols_wo_target:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    n_out = ((clean_df[col] < low) | (clean_df[col] > high)).sum()
    outlier_summary.append((col, int(n_out), float(n_out / len(clean_df) * 100)))

outlier_table = pd.DataFrame(outlier_summary, columns=["feature", "outliers", "outlier_pct"])
outlier_table.sort_values("outliers", ascending=False).head(10)

**Решение по выбросам:** для baseline-моделей агрессивное удаление выбросов не делаем, чтобы не потерять редкие, но валидные наблюдения. Смягчение влияния выбросов обеспечивается `median`-импутацией и `StandardScaler` в пайплайне.

## 3) Работа с фичами (feature engineering)

- Исходное число признаков: `24` (до добавления engineered-признаков).
- Добавленные признаки:
  - `NameLength` - длина имени животного,
  - `HasName` - бинарный признак наличия имени,
  - `FeePerPhoto` - отношение платы к числу фото (с защитой от деления на ноль).

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "Name" in df.columns:
        name_series = df["Name"].fillna("").astype(str)
        df["NameLength"] = name_series.str.len()
        df["HasName"] = (name_series.str.strip() != "").astype(int)

    if "Fee" in df.columns and "PhotoAmt" in df.columns:
        denom = df["PhotoAmt"].fillna(0).astype(float) + 1.0
        df["FeePerPhoto"] = df["Fee"].fillna(0).astype(float) / denom

    return df

clean_df = add_features(clean_df)
print("Shape after feature engineering:", clean_df.shape)

## 4) Визуализации зависимостей

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1) Распределение таргета
raw_df[DEFAULT_TARGET].value_counts().sort_index().plot(kind="bar", ax=axes[0, 0], title="AdoptionSpeed distribution")
axes[0, 0].set_xlabel("AdoptionSpeed")

# 2) Возраст по классам таргета
if "Age" in clean_df.columns:
    clean_df.boxplot(column="Age", by=DEFAULT_TARGET, ax=axes[0, 1])
    axes[0, 1].set_title("Age by AdoptionSpeed")
    axes[0, 1].set_xlabel("AdoptionSpeed")
else:
    axes[0, 1].set_visible(False)

# 3) Плата по классам таргета
if "Fee" in clean_df.columns:
    clean_df.boxplot(column="Fee", by=DEFAULT_TARGET, ax=axes[1, 0])
    axes[1, 0].set_title("Fee by AdoptionSpeed")
    axes[1, 0].set_xlabel("AdoptionSpeed")
else:
    axes[1, 0].set_visible(False)

# 4) Наличие имени vs таргет
if "HasName" in clean_df.columns:
    has_name_ct = pd.crosstab(clean_df["HasName"], clean_df[DEFAULT_TARGET], normalize="columns")
    has_name_ct.T.plot(kind="bar", ax=axes[1, 1], title="HasName share by class")
    axes[1, 1].set_xlabel("AdoptionSpeed")
else:
    axes[1, 1].set_visible(False)

plt.suptitle("", y=1.02)
plt.tight_layout()
plt.show()

## 5) Корректный split и контроль data leakage

Используем stratified split на `train/val/test`:
- `test_size=0.2`, `val_size=0.2` (от исходного датасета),
- `random_state=42` для воспроизводимости,
- стратификация по `AdoptionSpeed` для сохранения долей классов.

Контроль утечки данных:
- split выполняется **до** обучения моделей,
- любые трансформации (импутация, масштабирование, OHE) обучаются только на train в модельном пайплайне,
- в признаки не включается целевая переменная.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = make_train_val_test(clean_df)

print("Final split shapes:")
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "| y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "| y_test: ", y_test.shape)

print("\nTarget share by split (normalized):")
print("train:\n", y_train.value_counts(normalize=True).sort_index())
print("\nval:\n", y_val.value_counts(normalize=True).sort_index())
print("\ntest:\n", y_test.value_counts(normalize=True).sort_index())

## 6) Обоснование метрики качества

Для итогового сравнения моделей используем **Macro F1** как основную метрику:
- учитывает precision/recall одновременно,
- одинаково важна для всех классов,
- лучше подходит при потенциальном дисбалансе классов, чем чистая accuracy.

Дополнительно можно смотреть `accuracy` как вспомогательную метрику.